# 🤗 Laboratorio de modelos ya entrenados: Hugging Face en su empresa

**Diplomado en Inteligencia Artificial · UNAC · Clase 04 · Plantilla del aula**

---

### 🎯 Qué vamos a hacer

- **Clasificar las quejas de sus clientes** en las categorías que usted invente esta mañana, en español, sin entrenar nada y sin contratar a nadie.
- **Medir si un comentario es bueno, malo o tibio** y, de paso, aprender la pregunta que evita una demanda: *¿esta licencia me deja facturar con esto?*
- **Convertir un texto en voz** para la locución de un video, y devolver esa voz a texto — el círculo completo, como cuando se transcribe una reunión.
- **Ver de dónde salen los modelos grandes** (imagen y redacción gerencial) y por qué esos se llaman por teléfono en vez de correrlos en su computador.

### 🧭 Cómo se usa

**Usted no tiene que escribir código.** Aquí no venimos a volverlos programadores: venimos a que sepan mandar.

1. Corra las celdas **una por una** con el botón ▶️ que aparece a la izquierda de cada bloque gris, o todas de una vez con **Entorno de ejecución → Ejecutar todas**.
2. Lo **único** que usted cambia está en el **PANEL DE CONTROL** de más abajo. Nada más. Si algo se rompe, vuelve al panel, cambia el texto y corre otra vez.
3. Cada sección es **independiente**: si una falla, las demás siguen funcionando. Ninguna celda daña a la otra.

> ### 💡 Qué es un "modelo preentrenado" (dos minutos y seguimos)
>
> Entrenar un modelo de estos cuesta **millones de dólares** en electricidad, tarjetas gráficas e ingenieros. Alguien ya pagó esa cuenta —Google, Meta, Alibaba, un laboratorio universitario— y después **regaló el resultado** en un sitio que se llama Hugging Face: hoy hay más de un millón de modelos ahí, listos para descargar.
>
> Usted no entrena nada. Usted **usa** lo que otro pagó. Eso es todo lo que vamos a hacer hoy, y por eso funciona en un computador prestado con internet del aula.


## 1️⃣ ¿Qué textos de su empresa quiere que la máquina lea hoy?

Antes de empezar, usted decide con qué va a jugar el laboratorio: una queja de ejemplo, una reseña de ejemplo, y —lo más importante— **las categorías en las que quiere clasificar sus quejas**. Escríbalas como se dicen en su empresa, separadas por comas.

También decide qué frase se va a leer en voz alta y qué imagen le va a pedir al modelo de marketing.

### 🎛️ AQUÍ Y SOLO AQUÍ SE CAMBIAN COSAS

**Qué mirar en el resultado:** que el mensaje de confirmación repita exactamente sus categorías. Si escribió cuatro, deben salir cuatro.


In [ ]:
#@title 🎛️ PANEL DE CONTROL { display-mode: "both" }

# Una queja (PQR) suya, escrita como llega en la vida real:
pqr_de_ejemplo = "Llevo tres dias llamando y nadie contesta, y el domiciliario dejo la carne en la porteria" #@param {type:"string"}

# Una resena o comentario de cliente:
resena_de_ejemplo = "La atencion de la cajera fue excelente, pero la fila estaba eterna" #@param {type:"string"}

# Las categorias EN LAS QUE USTED quiere clasificar sus quejas (separadas por comas):
categorias_texto = "logistica y entregas, calidad del producto, atencion al cliente, precios y facturacion, infraestructura" #@param {type:"string"}

# La frase que va a leer la voz sintetica (locucion de video, audioguia, mensaje de la IPS):
texto_locucion = "Buenos dias. Le confirmamos que su pedido sale hoy antes del mediodia. Gracias por comprar con nosotros." #@param {type:"string"}

# La imagen de marketing que le vamos a pedir al modelo grande (seccion opcional):
prompt_de_imagen = "fotografia publicitaria de una fruteria moderna en Medellin, luz de la tarde, colores calidos" #@param {type:"string"}

categorias = [c.strip() for c in categorias_texto.split(",") if c.strip()]
print("Sus categorias (" + str(len(categorias)) + "):")
for c in categorias:
    print("   -", c)
print("\nPanel listo. No hay que tocar nada mas abajo.")


**Los textos de ejemplo que ya trae la plantilla.** Son quejas y comentarios inventados de un supermercado, una clínica y una alcaldía — el mismo estilo de lo que a usted le llega todos los días. Su queja del panel entra de primera en la lista.


In [ ]:
# Datos de ejemplo escritos aqui mismo: no se descarga ningun archivo, no hay rutas del computador.
PQRS_DEMO = [
    "El domiciliario llego dos horas tarde y la carne venia descongelada",
    "Me cobraron dos veces el mismo mercado y en la factura aparece un producto que no pedi",
    "La senora de la caja 4 fue amabilisima, pero el aire acondicionado del local no funciona",
    "Pedi la cita de control hace tres semanas y todavia no me han llamado del call center",
    "El bano del segundo piso de la sede lleva un mes fuera de servicio y huele muy mal",
    "Radique la solicitud en la ventanilla el 3 de julio y nadie me ha dado respuesta ni radicado",
    "El aguacate y el tomate venian golpeados, pero el precio del combo si esta muy bueno",
]

RESENAS_DEMO = [
    "Volveria a comprar sin pensarlo, quedaron muy bien",
    "Pesimo servicio, perdi toda la manana esperando",
    "El producto llego el dia que dijeron, sin novedades",
    "Nunca mas. Le mande tres correos y ni la molestia de contestar",
    "Buen precio, buena calidad, la recomiendo a los vecinos",
]

pqrs = [pqr_de_ejemplo] + PQRS_DEMO
resenas = [resena_de_ejemplo] + RESENAS_DEMO
print("Vamos a trabajar con", len(pqrs), "quejas y", len(resenas), "resenas.")


## 2️⃣ ¿Qué hay que instalar y por qué le fijamos la versión?

Colab ya trae Python 3.12 y casi todo lo demás. Solo instalamos la librería que habla con Hugging Face: `transformers`.

Le ponemos la versión **exacta** (`5.15.0`) a propósito. Esa librería saca una versión nueva casi cada semana, y en enero de 2026 salió la versión 5 que **borró funciones enteras** que los tutoriales viejos todavía usan. Si no fijamos el número, el notebook le puede servir hoy y romperse el mes entrante sin que usted haya tocado nada. Esta es la lección de gerente: *en producción no se usa "la última"; se usa la que uno probó.*

> ⏳ **La primera vez es la lenta.** Cada modelo se descarga la primera vez que se usa (entre 200 MB y 600 MB). Puede tardar uno o dos minutos. De ahí en adelante ya está en la máquina y es inmediato. Si 16 personas descargan al mismo tiempo, la red del salón sufre: tenga paciencia y no vuelva a correr la celda encima.

**Qué mirar en el resultado:** que diga `Python 3.12.x` y al final `Todo listo`. Los avisos amarillos de `pip` son normales, no son errores.


In [ ]:
!python --version
!pip install -q "transformers==5.15.0" sentencepiece

import warnings
warnings.filterwarnings("ignore")

import transformers
print("transformers:", transformers.__version__)
print("Todo listo. Siga con la siguiente celda.")


## 3️⃣ ¿Puedo clasificar 500 quejas en MIS categorías sin entrenar nada?

Sí. Y es la demostración que vale la clase entera.

Lo normal sería contratar a alguien seis semanas para etiquetar miles de quejas a mano y después "entrenar un modelo". Esto se llama **clasificación zero-shot** — *cero ejemplos* — y funciona al revés: usted le **escribe las categorías en español** y el modelo acomoda cada texto donde mejor encaje. Las categorías las puso usted en el panel de control hace un minuto.

El modelo es `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`: licencia MIT (uso comercial permitido), entiende 100 idiomas y **corre en el procesador normal**, sin tarjeta gráfica.

**Qué mirar en el resultado:** la columna `categoria_asignada` — lea las filas y pregúntese si usted habría clasificado igual. Y la columna `confianza`: cerca de 1 es "estoy seguro", cerca de 0,3 es "estoy adivinando entre varias".


In [ ]:
import pandas as pd

tabla_pqrs = None
try:
    from transformers import pipeline
    clasificador = pipeline(
        "zero-shot-classification",
        model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
        device=-1,  # -1 = procesador normal (CPU). No dependemos de tarjeta grafica.
    )
    salida = clasificador(pqrs, candidate_labels=categorias)
    if isinstance(salida, dict):
        salida = [salida]
    tabla_pqrs = pd.DataFrame({
        "queja": [t[:75] for t in pqrs],
        "categoria_asignada": [r["labels"][0] for r in salida],
        "confianza": [round(float(r["scores"][0]), 3) for r in salida],
    }).sort_values(["categoria_asignada", "confianza"], ascending=[True, False])
    display(tabla_pqrs)
except Exception as error:
    print("Esta seccion no corrio. Las demas si funcionan, siga adelante.")
    print("Detalle tecnico para copiarle a su Claude:", error)


**La foto que uno lleva a la junta** no es la tabla: es el conteo. Cuántas quejas cayeron en cada categoría.


In [ ]:
import matplotlib.pyplot as plt

try:
    conteo = tabla_pqrs["categoria_asignada"].value_counts()
    plt.figure(figsize=(8, 4))
    plt.barh(conteo.index[::-1], conteo.values[::-1], color="#EAAD05")
    plt.title("Quejas por categoria (clasificadas sin entrenar ningun modelo)")
    plt.xlabel("Cantidad de quejas")
    plt.ylabel("Categoria definida por usted")
    plt.tight_layout()
    plt.show()
except Exception as error:
    print("No hay tabla para graficar todavia. Corra primero la celda anterior.")
    print("Detalle tecnico:", error)


> ### 🏆 Deténgase un segundo en lo que acaba de pasar
>
> Usted **acaba de clasificar quejas sin entrenar nada**. No hubo datos históricos, no hubo etiquetadores, no hubo "proyecto de seis meses". Las categorías las escribió **usted, hace un minuto**, en el panel de control.
>
> Cambie las categorías arriba, vuelva a correr esta sección, y el modelo reclasifica todo. Eso es lo que en su empresa se traduce en: *"el lunes le pido a mi analista el reporte de PQRs por categoría, y las categorías las decido yo el viernes."*


## 4️⃣ ¿Este cliente quedó contento o quedó furioso?

Lo mismo, pero para la temperatura del comentario. Hay modelos entrenados específicamente con textos en español que dicen si algo es **positivo, negativo o neutro**. Con 2.000 reseñas eso deja de ser una anécdota y se vuelve un indicador que se puede graficar mes a mes.

Usamos `pysentimiento/robertuito-sentiment-analysis`, entrenado con miles de mensajes en español de varios países. Corre en el procesador normal, en segundos.

**Qué mirar en el resultado:** la columna `sentimiento`. Y fíjese en la reseña mixta ("la cajera fue excelente pero la fila eterna"): observe con cuánta confianza se decide. Ahí se ve que la máquina no entiende, *estima*.


In [ ]:
tabla_resenas = None
try:
    from transformers import pipeline
    analizador = pipeline(
        "sentiment-analysis",
        model="pysentimiento/robertuito-sentiment-analysis",
        device=-1,
    )
    salida_sent = analizador(resenas)
    nombres = {"POS": "positiva", "NEG": "negativa", "NEU": "neutra"}
    tabla_resenas = pd.DataFrame({
        "resena": [t[:75] for t in resenas],
        "sentimiento": [nombres.get(s["label"], s["label"]) for s in salida_sent],
        "confianza": [round(float(s["score"]), 3) for s in salida_sent],
    })
    display(tabla_resenas)
except Exception as error:
    print("Esta seccion no corrio. Las demas si funcionan, siga adelante.")
    print("Detalle tecnico para copiarle a su Claude:", error)


> ## ⚖️ La lección que vale más que la demo: la licencia
>
> Ese modelo que acabamos de usar funciona muy bien. **Y usted no debería montarlo en producción.**
>
> ¿Por qué? Porque en su ficha de Hugging Face **no declara licencia**, y sus autores advierten por escrito que los datos con los que se entrenó son de terceros y *"muchos de ellos son para uso no comercial"*. Traducción gerencial: sirve para aprender, para un piloto interno, para la clase de hoy. No para el producto que usted le factura a un cliente.
>
> **La regla que se llevan de este diplomado:** antes de que un modelo entre a un proceso de su empresa, alguien abre la página del modelo en Hugging Face y lee la casilla **"License"**. Son diez segundos. Los casos más comunes:
>
> | Etiqueta | Qué significa para su empresa |
> |---|---|
> | `apache-2.0`, `mit` | Verde. Uso comercial permitido, incluso dentro de un producto que usted cobre. |
> | `cc-by-nc-4.0` (el `nc` es lo grave) | Rojo. **No comercial.** Sirve para investigar y aprender, no para facturar. |
> | Licencias con nombre propio (*community license*, *non-commercial license*) | Amarillo: hay que leerlas. Varias son gratis solo por debajo de cierta facturación anual. |
> | Sin licencia declarada | Rojo prudente. Si nadie dice que se puede, asuma que no. |
>
> Más de la mitad de los modelos "abiertos" de moda no se pueden usar comercialmente. Casi nadie lo sabe. **Usted ya lo sabe** — y eso es exactamente la diferencia entre un usuario de IA y un gerente que manda sobre la IA.


## 5️⃣ ¿Puedo tener locución profesional en español sin pagar un estudio?

Esto es **texto a voz**. Se le escribe la frase y devuelve el audio, en español, con voz de persona.

Usamos `hexgrad/Kokoro-82M`: es el modelo de voz abierto más usado del mundo, pesa apenas 82 millones de parámetros —chiquito— y su licencia es **Apache-2.0: uso comercial permitido**. Voz en español: `ef_dora` (femenina); también existen `em_alex` y `em_santa` (masculinas).

Casos que salieron en el salón: la **locución del video de producto** sin contratar estudio, la **audioguía turística** en el hotel, el **mensaje grabado** que la IPS le pone al recordatorio de cita.

**Qué mirar en el resultado:** el reproductor de audio al final. Dele play y escuche. Después cambie `texto_locucion` en el panel de control y vuelva a correr esta celda.


In [ ]:
# Kokoro necesita un programa del sistema (espeak-ng) para pronunciar el espanol.
!apt-get -qq install -y espeak-ng > /dev/null 2>&1
!pip install -q kokoro soundfile

import numpy as np

ARCHIVO_VOZ = "locucion.wav"
try:
    import soundfile as sf
    from kokoro import KPipeline
    from IPython.display import Audio, display

    voz = KPipeline(lang_code="e")  # 'e' = espanol
    trozos = []
    for resultado in voz(texto_locucion, voice="ef_dora"):
        trozo = getattr(resultado, "audio", None)
        if trozo is None:
            trozo = list(resultado)[2]
        trozos.append(np.asarray(trozo, dtype="float32"))
    audio_final = np.concatenate(trozos)
    sf.write(ARCHIVO_VOZ, audio_final, 24000)
    print("Locucion lista:", round(len(audio_final) / 24000, 1), "segundos. Dele play:")
    display(Audio(ARCHIVO_VOZ))
except Exception as error:
    print("Esta seccion no corrio. Las demas si funcionan, siga adelante.")
    print("Detalle tecnico para copiarle a su Claude:", error)


## 6️⃣ ¿Y al revés? ¿La máquina me escribe lo que se dijo en la reunión?

Ahora cerramos el círculo: tomamos **el mismo audio que Kokoro acabó de crear** y se lo damos a un modelo que escucha y escribe. Si el texto que sale se parece al que usted escribió en el panel, el círculo **texto → voz → texto** funcionó.

Usamos `openai/whisper-small` (licencia MIT). Elegimos el "small" a propósito: es el que corre decente **sin tarjeta gráfica**, que es la situación de todos hoy. El hermano mayor se llama `openai/whisper-large-v3-turbo`, transcribe mucho mejor y en 99 idiomas, pero necesita tarjeta gráfica o llamarlo por API.

En su empresa esto transcribe **actas de junta**, **consultas médicas** y **llamadas del call center**. Y aquí va la advertencia que no es técnica: eso se hace **con consentimiento de quien habla**, y el audio de una historia clínica o de una llamada de cliente no se sube alegremente a un servidor ajeno. Ley 1581 de 2012.

**Qué mirar en el resultado:** compare la línea `Lo que escribio usted` con `Lo que entendio la maquina`. Va a haber diferencias en tildes y comas: eso es normal y es justo la conversación que hay que tener antes de automatizar un acta.


In [ ]:
import os

try:
    from transformers import pipeline
    if not os.path.exists("locucion.wav"):
        raise FileNotFoundError("Falta el audio: corra primero la seccion 5 (texto a voz).")

    transcriptor = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-small",
        device=-1,
    )
    try:
        dicho = transcriptor("locucion.wav", generate_kwargs={"language": "es"})
    except Exception:
        dicho = transcriptor("locucion.wav")

    print("Lo que escribio usted   :", texto_locucion)
    print("Lo que entendio la maquina:", dicho["text"].strip())
except Exception as error:
    print("Esta seccion no corrio. Las demas si funcionan, siga adelante.")
    print("Detalle tecnico para copiarle a su Claude:", error)


---

## 7️⃣ (OPCIONAL) ¿Y los modelos grandes, los que hacen imágenes?

> ### ⚠️ SECCIÓN OPCIONAL — requiere su token de Hugging Face
>
> Esta parte **la hace el docente en vivo**. Usted la corre con calma durante la semana. Si no le funciona hoy, no pasa absolutamente nada: las secciones 3 a 6 ya corrieron completas y sin token.

Hasta aquí todo corrió **dentro de su Colab**, gratis, en el procesador normal. Eso funciona porque los modelos que usamos son chiquitos: clasifican y deciden. Los que **crean** —una imagen, un informe de tres páginas, un video— pesan decenas de miles de millones de parámetros y **no caben** en la máquina que Google le presta. Esos se llaman por teléfono: uno le manda el pedido a un servidor ajeno y recibe el resultado.

La pregunta de gerente no es *"cuál modelo es mejor"*. Es *"¿esto lo corro en mi casa o lo llamo por teléfono, y quién ve mis datos en cada caso?"*.

**Su token, el de la Clase 03, va en la caja fuerte de Colab — nunca escrito dentro del notebook:**

1. Icono de la **llave 🔑** en la barra de la izquierda de Colab.
2. **Add new secret** (Agregar secreto nuevo).
3. Nombre: **`HF_TOKEN`** — exactamente así, en mayúsculas, sin espacios.
4. Value: pegue el token que creó en la Clase 03.
5. Active el interruptor **Notebook access** (acceso del notebook). Sin ese interruptor no sirve.

> 💰 **Lo que hay que saber antes de correrla:** el crédito gratis de la API de Hugging Face es de **unos 0,10 dólares al mes** (dato verificado en la página de precios, y ellos mismos advierten que puede cambiar). Se agota rapidísimo. **Si le sale un error 402, no hizo nada mal: se acabó el crédito del mes.** Es información de gerente, no una falla suya: así se ve el costo real de la IA generativa cuando se pasa de la demo al proceso.

**Qué mirar en el resultado:** la imagen. Y si no sale, el mensaje: le va a decir exactamente qué falta.


In [ ]:
token_hf = None
try:
    from google.colab import userdata
    token_hf = userdata.get("HF_TOKEN")
    print("Token encontrado en los secretos de Colab.")
except Exception:
    print("No encontre el secreto HF_TOKEN. Haga esto y vuelva a correr la celda:")
    print("  1) Icono de la llave (candado) en la barra izquierda de Colab")
    print("  2) Add new secret  ->  nombre exacto: HF_TOKEN")
    print("  3) Pegue el token de la Clase 03 en Value")
    print("  4) Active el interruptor 'Notebook access'")

if token_hf:
    try:
        from huggingface_hub import InferenceClient
        cliente = InferenceClient(api_key=token_hf)
        imagen = cliente.text_to_image(prompt=prompt_de_imagen, model="Tongyi-MAI/Z-Image-Turbo")
        display(imagen)
        print("Modelo: Tongyi-MAI/Z-Image-Turbo - licencia Apache-2.0, se puede facturar con esto.")
    except Exception as error:
        print("La API no respondio. Detalle:", error)
        print("Si dice 402: se agoto el credito gratis del mes (~0,10 USD). NO hizo nada mal.")
else:
    print("Seguimos sin imagen: no pasa nada, las secciones 3 a 6 ya corrieron sin token.")


## 8️⃣ ¿Quién me redacta el informe gerencial de todo esto?

Aquí hay un hallazgo que sirve para entender cómo se toman estas decisiones. Buscamos un modelito chiquito que **resuma en español** y servido por API… y **no existe**. Todos los resumidores disponibles por API están entrenados en inglés, y encima la versión 5 de `transformers` eliminó la función `pipeline("summarization")` justamente porque *"los modelos de chat modernos dan resultados de mucha mejor calidad"*.

Conclusión práctica: **para resumir en español se usa un LLM** — un modelo de conversación, de los que ustedes ya manejan. Dos caminos, y los dos están abajo:

- **Con token** (sigue siendo la sección opcional): un LLM abierto por la API de Hugging Face.
- **Sin token, y esto es la joya:** Colab regala un LLM propio. `from google.colab import ai` y ya. Sin token, sin tarjeta de crédito, sin cuenta de nube.

**Qué mirar en el resultado:** que el informe mencione **sus** categorías y **sus** números. No está inventando: está leyendo la tabla que salió arriba.


In [ ]:
# Armamos el insumo con los resultados reales del laboratorio.
insumo = "Resultados del laboratorio de modelos preentrenados:\n"
try:
    insumo += "- Quejas por categoria: " + str(tabla_pqrs["categoria_asignada"].value_counts().to_dict()) + "\n"
except Exception:
    insumo += "- La clasificacion de quejas no alcanzo a correr.\n"
try:
    insumo += "- Sentimiento de las resenas: " + str(tabla_resenas["sentimiento"].value_counts().to_dict()) + "\n"
except Exception:
    insumo += "- El analisis de sentimiento no alcanzo a correr.\n"
print(insumo)

if globals().get("token_hf"):
    try:
        from huggingface_hub import InferenceClient
        cliente = InferenceClient(api_key=token_hf)
        respuesta = cliente.chat.completions.create(
            model="Qwen/Qwen3.5-9B",
            messages=[{"role": "user", "content": "Resume en 5 vinetas para una junta directiva, en espanol de Colombia:\n" + insumo}],
        )
        print(respuesta.choices[0].message.content)
    except Exception as error:
        print("La API no respondio (si dice 402, se agoto el credito del mes). Detalle:", error)


**Y ahora sin token, para todos.** Esta celda usa el LLM que Google incluye dentro de Colab. No pide nada: ni token, ni tarjeta, ni cuenta de nube. Es el cierre más cómodo si la sección del token no le funcionó.


In [ ]:
if "insumo" not in globals():
    insumo = "El laboratorio corrio con quejas de ejemplo clasificadas en categorias definidas por el usuario."

try:
    from google.colab import ai
    informe = ai.generate_text(
        "Redacta en tres parrafos, en espanol de Colombia y en lenguaje de negocio (sin tecnicismos), "
        "el informe gerencial de estos resultados. Cierra con una recomendacion concreta:\n" + insumo
    )
    print(informe)
except Exception as error:
    print("El LLM de Colab no respondio ahora. Vuelva a intentar en un rato.")
    print("Detalle tecnico para copiarle a su Claude:", error)


---

## 9️⃣ 🧭 El mapa completo: qué modelo para qué, y con qué licencia

Guárdese esta tabla. Es la que va a abrir el día que en su empresa alguien diga *"deberíamos usar IA para esto"*. Todo verificado en agosto de 2026 — y los nombres de los modelos cambian rápido, así que confírmelos antes de firmar algo.

| Lo que usted quiere hacer | Modelo estrella (nombre exacto) | Licencia | Dónde corre |
|---|---|---|---|
| Clasificar quejas, correos, tickets en **sus** categorías | `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` | **MIT** ✅ comercial | Su Colab, sin tarjeta gráfica |
| Medir sentimiento en español | `pysentimiento/robertuito-sentiment-analysis` | ⚠️ **sin licencia declarada** — solo aprender | Su Colab, sin tarjeta gráfica |
| Lo mismo, por API y sin descargar nada | `cardiffnlp/twitter-xlm-roberta-base-sentiment` | ⚠️ no declarada | API y Colab |
| Texto → voz en español | `hexgrad/Kokoro-82M` (voces `ef_dora`, `em_alex`) | **Apache-2.0** ✅ comercial | Su Colab y API |
| Voz → texto (transcribir reuniones) | `openai/whisper-small` (CPU) · `openai/whisper-large-v3-turbo` (el bueno) | **MIT** ✅ comercial | Small en su Colab; turbo con tarjeta gráfica o API |
| Leer facturas, planos, PDFs escaneados | `zai-org/GLM-OCR` (0,9B) · `Qwen/Qwen3-VL-4B-Instruct` | **MIT** / **Apache-2.0** ✅ | Colab con tarjeta gráfica T4 |
| Generar imagen de marketing o render | `Tongyi-MAI/Z-Image-Turbo` | **Apache-2.0** ✅ comercial | API (cabe justo en una T4) |
| Resumir o redactar en español | *no hay modelo chico servido* → un LLM: `Qwen/Qwen3.5-9B` por API, o `google.colab.ai` | según el LLM | API o el Colab de Google |
| Generar video | nada de esto cabe en Colab gratis | varias no comerciales | Solo API o servicio pago |

### 🚨 La trampa de las licencias, con un ejemplo que se ve solo

Dos traductores. Hacen lo mismo. Uno lo deja facturar y el otro no:

| Modelo | Licencia | ¿Puede usarlo en su empresa? |
|---|---|---|
| `google/madlad400-3b-mt` — 419 idiomas | **Apache-2.0** | ✅ **Sí.** Traduzca pliegos, anexos, contratos, y factúrelo. |
| `facebook/nllb-200-distilled-600M` — 196 idiomas | **CC-BY-NC-4.0** | ❌ **No.** Es *no comercial*, y su propia ficha dice que **"no fue liberado para despliegue en producción"**. |

El que sale primero en los blogs y en los tutoriales es el segundo. Elegir el equivocado no se nota hasta que llega un abogado. **Otros de la lista negra para uso comercial:** `facebook/mms-tts-spa`, `mistralai/Voxtral-4B-TTS-2603`, `black-forest-labs/FLUX.2-dev`, `FLUX.2-klein-9B`, XTTS v2 de Coqui.

### 🗺️ Para seguir explorando por su cuenta

- **https://huggingface.co/tasks** — el catálogo por tarea, en lenguaje humano: "clasificar texto", "transcribir audio", "leer documentos". Entra, elige la tarea, y ve los modelos ordenados por uso. Es el mejor sitio para empezar.
- **https://huggingface.co/spaces?sort=trending** — los **Spaces**: modelos con botones, sin código. Sube el archivo, aprieta, sale el resultado. Perfecto para probar antes de meter a nadie a programar.
- ⏱️ **Aviso sobre los Spaces:** la tarjeta gráfica gratis de los Spaces son **5 minutos al día** con cuenta creada (y solo **2 minutos** sin iniciar sesión). Alcanza para dos o tres pruebas, no para una tarde de trabajo. Para trabajar de verdad: Colab.

### 💡 Las tres cosas que un gerente separa y casi todo el mundo confunde

1. **Licencia del modelo** — ¿me deja facturar con esto?
2. **Costo de cómputo** — "abierto" no es "gratis de operar". Un modelo Apache-2.0 de 20 mil millones de parámetros sigue costando tarjetas gráficas todos los meses.
3. **Quién ve mis datos** — un modelo que corre "en su Colab" también está subiendo el archivo a un servidor de Google. Y la API enruta a proveedores de terceros. Datos de pacientes, nómina, contratos o clientes: anonimizados o no van.


---

## 🏁 Ahora con SUS datos

Esta plantilla trae quejas y reseñas inventadas. Lo que se lleva a su empresa es el mismo notebook con **sus** textos. Hay dos caminos y ninguno requiere programar:

**Camino 1 — el más rápido (funciona hoy).** Vuelva al **PANEL DE CONTROL** y pegue una queja real de su empresa en `pqr_de_ejemplo`, un comentario real en `resena_de_ejemplo`, y **sus** categorías de verdad en `categorias_texto`. Corra otra vez las secciones 3 y 4. Eso es todo. Ya está clasificando lo suyo.

**Camino 2 — con un archivo de Excel o CSV entero.** Cuando quiera correr 500 quejas de una sentada:

1. Exporte de su sistema un archivo `.csv` con **una sola columna de texto** (por ejemplo la columna se llama `queja`) — una fila por queja.
2. En Colab, en la barra de la izquierda, entre a **Archivos → Subir al almacenamiento de la sesión** y suba su `.csv`. (También se puede hacer por código con `files.upload()` de Colab, pero el botón es más fácil y hace exactamente lo mismo.)
3. Pídale a su Claude que cambie la lista de ejemplo por la lectura de su archivo. El prompt está aquí abajo, listo para copiar.

> ⚠️ **Antes de subir nada:** este archivo se va a un servidor de Google, y en la sección opcional los textos se van a un servidor de terceros. Nombres de pacientes, cédulas, números de contrato, nómina: **eso se borra o se reemplaza antes de subirlo.** Habeas data, Ley 1581 de 2012. La regla del salón: en el laboratorio se trabaja con datos anonimizados, siempre.

> 💾 **Y guarde su trabajo.** Todo lo que se instaló y todo lo que subió **se pierde** cuando se cierra la sesión de Colab. Su entregable es **Archivo → Guardar una copia en Drive**, y el PDF con **Archivo → Imprimir → Guardar como PDF**.


### 🤖 Pregúntele a su Claude

Copie y pegue cualquiera de estos tres. Y antes, arrastre este mismo `.ipynb` a la conversación: Claude lo lee completo y sabe exactamente de qué le está hablando.

**1. Para conectar su archivo:**
> *"Te adjunto el notebook `5_Laboratorio_Modelos_Preentrenados_HF.ipynb` de mi diplomado. Quiero correr la sección 3 (clasificación zero-shot) con mi propio archivo en vez de la lista de ejemplo. Mi CSV se llama `quejas.csv` y la columna con el texto se llama `queja`; tiene unas 500 filas. Dame la celda de reemplazo, corta, con manejo de errores amable en español, y dime dónde exactamente la pego. No cambies nada más del notebook."*

**2. Para adaptar las categorías a su negocio:**
> *"Trabajo en [describa su empresa: una IPS de segundo nivel / un supermercado de barrio con tres sedes / contratación estatal en una alcaldía]. Proponme entre 5 y 8 categorías para clasificar las PQRs que nos llegan, con el criterio de que cada categoría le sirva a un área distinta para tomar una decisión. Explícame por qué cada una y qué decisión habilita."*

**3. Para el modelo correcto y la licencia correcta:**
> *"Necesito [transcribir las actas de junta / leer 300 facturas en PDF y sacar NIT, fecha, base e IVA / generar las piezas de campaña del mes]. Busca en Hugging Face qué modelo me conviene, dime la licencia exacta y si me deja usarlo comercialmente, cuánta memoria de tarjeta gráfica necesita, y si me sale mejor correrlo en Colab o llamarlo por API. Advíérteme si es CC-BY-NC."*


---

<div align="center">

**Diplomado en IA · UNAC · Clase 04**

*Si algo falla, copie el error completo y péguelo a su Claude: él sabe de este notebook.*

</div>
